# 05 · Gemma 4 31B 제로샷 (A100, 학습 없음)

앙상블의 Gemma 자리에 들어가는 확률 파일을 만듭니다. 이미지 토큰 예산만 다른 세 설정을 각각 한 번씩 실행합니다.

| SETTING | 이미지 토큰 | 프롬프트 | valid (원본) | 원본 실험 |
|---|---|---|---|---|
| `"560"` | 560 | OCR 강조 | 645/672 | EXP-120 |
| `"280"` | 280 | 짧은 직접형 | 635/672 | EXP-122 |
| `"1120"` | 1120 | OCR 강조 | 648/672 | EXP-124 |

- A100 40GB에는 bf16(약 62GB)이 안 들어가서 **4bit(nf4)** 로 로드
- thinking 끔, 답을 생성하지 않고 a~d 토큰 확률을 직접 읽음
- 소요: 560 약 1.5시간, 280 약 1시간, 1120 약 3.5시간 (test 6,714 기준). 끊기면 5·6번 셀만 다시 실행하면 이어서 진행

In [ ]:
# ===== 0. 패키지 설치 (실행 후 '런타임 → 세션 다시 시작') =====
!pip install -q -U transformers accelerate peft bitsandbytes
!pip uninstall -y -q torchao          # 구버전 torchao가 peft와 충돌
!apt-get install -y -qq p7zip-full > /dev/null

In [ ]:
# ===== 1. 드라이브 · 데이터 =====
import os, glob, shutil, subprocess, time
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
OUT_DIR = f"{SHARE_DIR}/repro"; os.makedirs(OUT_DIR, exist_ok=True)
DATA_ROOT = "/content/data"
cnt = lambda d: len(glob.glob(f"{DATA_ROOT}/{d}/*.jpg"))
if cnt("train") < 6714 or cnt("test") < 6714:
    shutil.copy(glob.glob(f"{SHARE_DIR}/ssafy-16-2-ai*.zip")[0], "/content/data.zip")
    shutil.rmtree(DATA_ROOT, ignore_errors=True); os.makedirs(DATA_ROOT)
    subprocess.run(["7z","x","-y","-pocrisnotenough",f"-o{DATA_ROOT}","/content/data.zip"], check=True, stdout=subprocess.DEVNULL)
print("이미지 수 → train:", cnt("train"), "test:", cnt("test"))
resolve = lambda p: p if os.path.isabs(str(p)) else os.path.join(DATA_ROOT, str(p))
valid_df = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv"); valid_df["path"] = valid_df["path"].map(resolve)
test_df  = pd.read_csv(f"{DATA_ROOT}/test.csv"); test_df["path"] = test_df["path"].map(resolve)
print("valid", valid_df.shape, "test", test_df.shape)

In [ ]:
# ===== 2. 모델 로드 (약 5~10분, 다운로드 약 62GB) =====
import torch, torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
Image.MAX_IMAGE_PIXELS = None

MODEL_ID = "google/gemma-4-31B-it"
SETTING = "560"          # "560", "280", "1120" 중 하나
SOFT_TOKENS = int(SETTING)
TAG = f"gemma{SETTING}"
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if mem_gb > 70:
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda", attn_implementation="sdpa")
else:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="cuda", attn_implementation="sdpa")
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left", max_soft_tokens=SOFT_TOKENS)
tok = processor.tokenizer
print(type(model).__name__, "| GPU", round(mem_gb), "GB | 사용", round(torch.cuda.memory_allocated()/1e9,1), "GB")

In [ ]:
# ===== 3. 프롬프트 · 설정 점검 (생성 결과를 눈으로 확인) =====
def prompt(r):
    if SETTING == "280":   # 짧은 직접형 (EXP-122)
        return (f"{r['question']}\n" + "".join(f"{l}. {r[l]}\n" for l in "abcd") +
                "\n알맞은 답의 기호(a, b, c, d) 하나만 출력하세요.")
    return (f"이미지를 자세히 보고 질문에 답하세요. 이미지 속 글자를 정확히 읽는 것이 중요합니다.\n\n"
            f"질문: {r['question']}\n" + "".join(f"{l}. {r[l]}\n" for l in "abcd") +
            "\n정답 선지의 알파벳(a, b, c, d) 한 글자만 출력하세요.")

def build(r):
    img = Image.open(r["path"]).convert("RGB")
    msgs = [{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":prompt(r)}]}]
    try: text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[img], return_tensors="pt").to(model.device), text

for _, r in valid_df.head(3).iterrows():
    inp, text = build(r)
    with torch.inference_mode():
        out = model.generate(**inp, max_new_tokens=12, do_sample=False)
    print("정답", r["answer"], "| 생성:", repr(tok.decode(out[0, inp["input_ids"].shape[1]:], skip_special_tokens=False)), "| 입력 토큰", inp["input_ids"].shape[1])
print("\n프롬프트 끝부분:", repr(text[-120:]))
print("→ 생성 결과가 'a', 'b' 처럼 글자 하나로 시작하면 정상. 생각(thought) 태그가 먼저 나오면 알려주세요.")

In [ ]:
# ===== 4. 로짓 채점 + 속도 측정 (30문항) =====
def ids_for(ch):
    s = set()
    for x in [ch, ch.upper(), " "+ch, " "+ch.upper()]:
        t = tok.encode(x, add_special_tokens=False)
        if len(t) == 1: s.add(t[0])
    return sorted(s)
CHOICE_IDS = [ids_for(c) for c in "abcd"]
print("선지 토큰:", CHOICE_IDS)

@torch.inference_mode()
def score(r):
    inp, _ = build(r)
    try: lg = model(**inp, logits_to_keep=1).logits[0, -1].float()
    except TypeError: lg = model(**inp).logits[0, -1].float()
    return F.softmax(torch.stack([torch.logsumexp(lg[i], 0) for i in CHOICE_IDS]), 0).cpu().numpy()

t0 = time.time(); sub = valid_df.head(30)
pz = np.array([score(r) for _, r in sub.iterrows()])
print(f"30문항 정답률 {(pz.argmax(1) == sub.answer.map(dict(zip('abcd',range(4)))).values).mean():.2f} | {(time.time()-t0)/30:.2f}초/문항")
print(f"→ test 6,714문항 예상: {(time.time()-t0)/30*6714/60:.0f}분")

In [ ]:
# ===== 5. valid 672 (중간 저장·재개) =====
P = ["prob_a","prob_b","prob_c","prob_d"]
def run(df, out_csv, every=50):
    done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["id"]+P)
    todo = df[~df.id.astype(str).isin(set(done.id.astype(str)))]; rows = []; t0 = time.time()
    print(f"완료 {len(done)} / 남음 {len(todo)}")
    for n, (_, r) in enumerate(todo.iterrows(), 1):
        try: p = score(r)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); p = np.full(4, .25); print("OOM", r.id)
        rows.append([r.id, *p])
        if n % every == 0 or n == len(todo):
            done = pd.concat([done, pd.DataFrame(rows, columns=done.columns)], ignore_index=True); done.to_csv(out_csv, index=False); rows = []
            el = time.time()-t0; print(f"{n}/{len(todo)} {el/n:.2f}s/문항, 남은 {el/n*(len(todo)-n)/60:.0f}분")
    return done.set_index("id")

vp = run(valid_df, f"{OUT_DIR}/{TAG}_valid_probs.csv")
ids = valid_df.id.values; ans = valid_df.answer.map(dict(zip("abcd",range(4)))).values
g = vp.loc[ids, P].values.astype(float)
print(f"Gemma {SETTING} 단독 valid: {(g.argmax(1)==ans).sum()}/672 (원본: 560→645, 280→635, 1120→648)")

## 6. test 6,714 (끊기면 이 셀만 다시 실행 → 이어서 진행)

In [ ]:
tp = run(test_df, f"{OUT_DIR}/{TAG}_test_probs.csv", every=100)
print(tp.shape)